# GMM V0: peak EEM và experiment các nhóm đủ dữ liệu

Notebook triển khai phân rã **cộng** bằng Gaussian với mask training, khởi tạo tránh plateau zero, tâm trong miền đo, amplitude NNLS và prior mềm theo vùng phổ. Nó chạy ba ablation trên đúng các nhóm đủ dữ liệu của experiment V0 trước đó.

**Câu hỏi:** sửa mô hình có cải thiện reconstruction và vị trí cực đại không; prior/độ nghiêng có giúp hay làm mất tính linh hoạt; component có ổn định qua khởi tạo và thời gian không?

B/T/M/A/C/UV là **tên vùng giả thuyết**, không phải kết luận danh tính hóa học. Peak của tổng EEM cũng không nhất thiết trùng tâm một component khi các tín hiệu chồng lấn.

- Mô hình: `src/eem_water_quality/features/improved_v0.py`.
- Experiment và cache: `scripts/diagnostics/run_improved_v0.py`.
- Output: `runs/diagnostics/v0_improved_groups/`.

Chạy **Run All**. Các fit được cache theo dữ liệu train, cấu hình, mã solver và phiên bản thư viện. Notebook đã chạy sẽ có bảng/figure; chạy lại dùng cache khi các đầu vào không đổi.
Historical V0 theo group là tùy chọn: nếu chưa chạy notebook V0 gốc theo group, phần so sánh cũ hiển thị thiếu; benchmark cải tiến vẫn chạy độc lập.


In [ ]:
from pathlib import Path
import os, sys, copy, json
os.environ.setdefault("MPLCONFIGDIR", "/tmp/eem-v0-matplotlib")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/processed/eem.npy").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts/diagnostics"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, HTML, FileLink
from scipy.optimize import linear_sum_assignment
from eem_water_quality.features.improved_v0 import REGIONS, fit_improved_v0
from run_improved_v0 import DEFAULT_CONFIG, run_experiment, summarize, slug
OUT = ROOT / "runs/diagnostics/v0_improved_groups"
OUT.mkdir(parents=True, exist_ok=True)
X = np.asarray(np.load(ROOT / "data/processed/eem.npy"), float)
samples = pd.read_csv(ROOT / "data/processed/samples.csv")
with np.load(ROOT / "data/processed/wavelengths.npz") as w:
    ex, em = w["excitation"], w["emission"]
group_manifest = samples.groupby("Group").agg(n_samples=("Point", "size"), n_months=("Month", "nunique")).reset_index()
group_manifest = group_manifest[(group_manifest.n_samples >= 8) & (group_manifest.n_months >= 8)].reset_index(drop=True)
assert X.shape == (len(samples), len(em), len(ex))
assert len(group_manifest) > 0
print(f"{X.shape}; Ex {ex.min():.0f}–{ex.max():.0f}, Em {em.min():.0f}–{em.max():.0f} nm")
display(group_manifest)

## Mô hình và những thay đổi so với V0

\[
Y_n(s)=a_{n,bg}\phi_{bg}(s)+\sum_{k=1}^{6}a_{nk}\phi_k(s;\mu_k,\sigma_k,\rho_k)+\epsilon_n(s),\quad a_{nk}\ge0.
\]

Gaussian được chuẩn hóa trên miền quan sát **training** để xác định scale amplitude. Giá trị âm nhỏ trong EEM được giữ như nhiễu đo; tín hiệu kỳ vọng không âm. Không ép tổng reconstruction bằng đúng tổng EEM. Nền là offset cộng, **không phải posterior “pixel thuộc nền”**.

Loss là bình phương sai số cân bằng theo norm mỗi EEM (floor percentile 5 từ train), cộng penalty khoảng cách tâm tới vùng tham chiếu và penalty rho. Bước cập nhật shape dùng Adam; mỗi checkpoint giải lại amplitude bằng NNLS, đánh giá và lưu đúng cùng một state. Trong warm-up chỉ cập nhật width/amplitude. Tâm bị chặn theo miền đo, không bị chặn cứng vào hộp sinh hóa.

Ba biến thể có **cùng rank 6, nền, khởi tạo vùng và ngân sách**:

| Biến thể | Prior tâm | Covariance |
|---|---|---|
| `diag_no_prior` | Không penalty | rho=0 |
| `diag_soft_prior` | Mềm | rho=0 |
| `rho_soft_prior` | Mềm | \|rho\|≤0.3 và penalty |

`diag_no_prior` vẫn khởi tạo từ cùng vùng để chỉ thay penalty. UV giữ ứng viên quanh Ex~230/Em~340 được quan sát trong audit. Model có thể tắt amplitude; việc đặt 6 ứng viên không có nghĩa cả 6 chất đều tồn tại.

**Phạm vi:** đây là bước Gaussian được kiểm soát trước khi triển khai nhiều excitation lobes dùng chung emission profile. Chưa có mô hình multi-lobe hay blank/noise likelihood đầy đủ.

In [ ]:
CONFIG = copy.deepcopy(DEFAULT_CONFIG)
# Có thể tăng steps/starts cho lượt xác nhận; cache tự đổi khi cấu hình thay đổi.
# Defaults: full=400 steps × 3 starts; mỗi temporal fold=300 steps × 2 starts.
RUN_EXPERIMENT = True
print(json.dumps(CONFIG, indent=2))
display(pd.DataFrame(REGIONS, columns=["hypothesis", "Ex_low", "Ex_high", "Em_low", "Em_high"]))

## Sanity check: khôi phục peak đã biết

Dữ liệu tổng hợp có B/T chồng lấn, một peak humic, nền, nhiễu cộng và tam giác bị mask. Ground truth chỉ tồn tại ở thí nghiệm này. Train 24 mẫu, kiểm tra 6 mẫu mới với shape cố định. Đây là sanity check khớp họ Gaussian, chưa chứng minh tính đúng trên mọi phổ phi Gaussian.

In [ ]:
rng = np.random.default_rng(7)
sx, sy = np.arange(240., 371., 5), np.arange(290., 481., 3)
xx, yy = np.meshgrid(sx, sy)
truth_centers = np.array([[275, 315], [280, 343], [340, 435]])
truth_widths = np.array([[9, 9], [11, 12], [20, 25]])
truth_shapes = np.array([np.exp(-0.5*((xx-c[0])/w[0])**2-0.5*((yy-c[1])/w[1])**2)
                         for c,w in zip(truth_centers, truth_widths)])
synthetic = np.einsum("nk,kij->nij", rng.uniform(.1, 3, (30,3)), truth_shapes) + .015
synthetic += rng.normal(0, .005, synthetic.shape)
synthetic_mask = yy > xx+20
synthetic[:, ~synthetic_mask] = 0
synthetic_regions = (("B",270,280,305,325),("T",275,290,330,355),("C",325,355,415,455))
synthetic_fit = fit_improved_v0(synthetic[:24], sx, sy, regions=synthetic_regions,
                               mask=synthetic_mask, n_starts=2, max_steps=300)
cost = np.linalg.norm(truth_centers[:,None]-synthetic_fit.centers[None], axis=2)
ri, ci = linear_sum_assignment(cost)
synthetic_result = pd.DataFrame({"peak": [synthetic_regions[i][0] for i in ri],
    "true_Ex": truth_centers[ri,0], "true_Em": truth_centers[ri,1],
    "fit_Ex": synthetic_fit.centers[ci,0], "fit_Em": synthetic_fit.centers[ci,1],
    "center_error_nm": cost[ri,ci]})
synthetic_pred = synthetic_fit.reconstruct(synthetic[24:])
synthetic_error = np.linalg.norm((synthetic_pred-synthetic[24:])[:,synthetic_mask]) / np.linalg.norm(synthetic[24:,synthetic_mask])
print(f"Held-out relative RMSE: {synthetic_error:.4f}")
display(synthetic_result.round(3))
assert synthetic_result.center_error_nm.max() < 8
assert synthetic_error < .1
synthetic_result.to_csv(OUT / "synthetic_recovery.csv", index=False)
(OUT / "synthetic_metrics.json").write_text(json.dumps({"relative_rmse": float(synthetic_error)}))

## Protocol các nhóm đủ dữ liệu và chạy experiment

Chọn nhóm có ≥8 mẫu và ≥8 tháng. Chia nửa đầu các tháng quan sát làm train ban đầu; nửa sau chia ba block liên tiếp (nhóm đủ 12 tháng có lịch dưới đây). Ba fold mở rộng:

1. Train tháng 1–6, test 7–8.
2. Train tháng 1–8, test 9–10.
3. Train tháng 1–10, test 11–12.

Mọi mẫu trong cùng tháng đi cùng phía. Mask, peak shape, scale của loss và hồi quy đều được fit từ train. Test chỉ infer amplitude trên shape cố định. EEM test là đầu vào để suy ra chất lượng nước cùng kỳ; đây **không phải dự báo trước khi đo EEM**.

Ridge alpha=10 cố định, `log1p(amplitudes)` và StandardScaler fit từ train; dùng tất cả component và nền. Baseline gồm mean train, tổng cường độ và cực đại các vùng tham chiếu. Không chọn component/rank bằng target test.

V0 lịch sử đã fit toàn bộ dữ liệu, chỉ xuất hiện ở bảng **in-sample**. Nó có 4 Gaussian còn mô hình mới có 6, nên so sánh với V0 cũ đánh giá cả gói thay đổi, không tách được riêng tác động của rank. Ba biến thể mới mới là ablation cùng rank.

Mask `training_nonzero` chỉ là proxy cho miền đã được xử lý, chưa được xác minh bằng metadata phòng thí nghiệm. Các giá trị test ở ô không từng được quan sát trong train được thống kê riêng.

In [ ]:
if RUN_EXPERIMENT:
    results = run_experiment(OUT, CONFIG, group_col="Group")
else:
    if not (OUT / "prediction_summary.csv").exists():
        raise FileNotFoundError("Chưa có kết quả; bật RUN_EXPERIMENT=True")
    results = summarize(OUT)
display(results["overview"].round(4))

## Reconstruction: mỗi nhóm được tính trọng số ngang nhau

Relative RMSE = norm(residual)/norm(observed), tính trên miền hợp lệ. Bảng overview lấy median trong mỗi nhóm rồi median giữa nhóm. Bảng tương lai không có historical V0.

Khoảng cách peak là sai lệch cực đại **của tổng EEM**; cần xem thêm residual và cực đại theo vùng vì hai đỉnh gần bằng nhau có thể đổi thứ tự.

In [ ]:
reconstruction = pd.read_csv(OUT / "reconstruction_summary.csv")
full_table = reconstruction.query("evaluation == 'in_sample'").pivot(index="Group", columns="variant", values="median_relative_rmse")
future_table = reconstruction.query("evaluation == 'future_months'").pivot(index="Group", columns="variant", values="median_relative_rmse")
print("IN-SAMPLE")
display(full_table.round(4))
print("FUTURE MONTHS")
display(future_table.round(4))
display(Image(filename=str(OUT / "figures/reconstruction_comparison.png")))
comparison = pd.DataFrame(index=full_table.index)
comparison["old_RMSE"] = full_table.get("historical_V0", pd.Series(np.nan, index=full_table.index))
comparison["diag_prior_RMSE"] = full_table["diag_soft_prior"]
comparison["relative_improvement_pct"] = 100*(1-comparison.diag_prior_RMSE/comparison.old_RMSE)
display(comparison.round(2))
print("Số nhóm tốt hơn V0 cũ (in-sample):", int((comparison.relative_improvement_pct>0).sum()), "/", len(comparison))

## Tác động của prior và rho trên dữ liệu tương lai

Giá trị âm của delta nghĩa là tốt hơn. Không dùng kết quả này để chỉnh lại tham số rồi vẫn gọi chính các fold này là test độc lập. Nếu tiếp tục chọn cấu hình, cần tập xác nhận khác/nested evaluation.

In [ ]:
ablation = pd.DataFrame(index=future_table.index)
ablation["prior_minus_no_prior"] = future_table.diag_soft_prior - future_table.diag_no_prior
ablation["rho_minus_diag"] = future_table.rho_soft_prior - future_table.diag_soft_prior
display(ablation.round(4))
regional = pd.read_csv(OUT / "regional_reconstruction.csv")
region_summary = (regional[regional.split.ne("full")]
    .groupby(["Group", "variant", "region"])[["median_relative_rmse", "median_window_max_distance_nm"]].median()
    .groupby(["variant", "region"]).median())
display(region_summary.round(3))

## Tâm, độ rộng và mức hoạt động

`inside_reference_window` là mức phù hợp prior, **không phải accuracy sinh hóa**. `fraction_share_over_1pct` chỉ là mô tả amplitude, không phải kiểm định phát hiện tín hiệu. Seed không có local maximum vẫn có thể là giả thuyết cho một vai phổ; trường `initial_local_max_supported` ghi rõ trường hợp này.

In [ ]:
parameters = pd.read_csv(OUT / "component_parameters.csv")
GROUP = "han_corridor"
VARIANT = "diag_soft_prior"
display(parameters[(parameters.Group == GROUP) & (parameters.variant == VARIANT)].round(3))
geometry_summary = parameters.groupby("variant").agg(
    fraction_inside_prior=("inside_reference_window", "mean"),
    median_axis_ratio=("axis_ratio_nm", "median"),
    max_abs_rho=("rho", lambda s: s.abs().max()),
    median_prior_distance_nm=("prior_distance_nm", "median"))
display(geometry_summary.round(3))
assert parameters.center_Ex_nm.between(ex.min(), ex.max()).all()
assert parameters.center_Em_nm.between(em.min(), em.max()).all()

## Figure 

Mặc định xem mẫu đầu tiên của han_corridor, cùng mẫu ở cả hai model. Các ô không quan sát được hiển thị trống. Thang màu intensity dùng chung; residual có thang riêng. Các HTML của cả các nhóm đủ dữ liệu được lưu offline cùng output.

In [ ]:
plot_path = OUT / "figures" / f"{slug(GROUP)}.html"
display(HTML(plot_path.read_text()))

## Độ ổn định qua khởi tạo và thời gian

Ghép component bằng Hungarian matching trên cosine surface trên miền chung, không so tên C1/C2 một cách máy móc. Seed stability bỏ nghiệm tốt nhất khi tổng hợp. Temporal stability so các fit train với fit toàn bộ dữ liệu **chỉ để mô tả**, không dùng full fit để tạo feature CV.

Cần đọc cosine cùng dịch chuyển tâm: surface rộng có thể cosine cao dù tâm dịch. Component gần như không hoạt động cũng có thể khó xác định.

In [ ]:
seed_stability = pd.read_csv(OUT / "seed_stability.csv")
seed_summary = seed_stability[seed_stability.start != seed_stability.best_start].groupby("variant").agg(
    median_min_cosine=("cosine_min", "median"),
    worst_cosine=("cosine_min", "min"),
    median_max_center_shift_nm=("center_shift_nm_max", "median"))
display(seed_summary.round(3))
temporal = pd.read_csv(OUT / "temporal_stability.csv")
display(temporal.groupby("variant").agg(median_cosine=("cosine", "median"),
    cosine_p10=("cosine", lambda s:s.quantile(.1)),
    median_center_shift_nm=("center_shift_nm", "median")).round(3))
fit_runs = pd.read_csv(OUT / "fit_runs.csv")
display(results["convergence"])
print("Các run hết ngân sách chưa thỏa tiêu chí hội tụ:", int((~fit_runs.converged).sum()), "/", len(fit_runs))

## Chất lượng nước: end-to-end temporal evaluation

R² OOF được tính từ toàn bộ dự đoán tương lai ghép lại trong mỗi nhóm, không lấy trung bình R² trên các fold chỉ có vài mẫu. `skill_vs_train_mean` so SSE với dự đoán mean thật sự chỉ học từ training của từng fold. Baseline này có thể có R² OOF âm khi dữ liệu dịch chuyển theo mùa.

Mục tiêu chính vẫn là phân rã phổ. R² cao không xác nhận đúng danh tính peak, R² thấp không tự chứng minh phân rã sai.

In [ ]:
predictive = pd.read_csv(OUT / "prediction_summary.csv")
display(predictive.groupby(["variant", "target"])[["R2_oof", "skill_vs_train_mean", "MAE"]].median().round(3))
display(predictive[predictive.Group == GROUP].sort_values(["target", "variant"]).round(3))
predictions = pd.read_csv(OUT / "predictions.csv")
fig, axes = plt.subplots(1, 3, figsize=(13,4), layout="constrained")
for ax, target in zip(axes, ["BOD", "COD", "TOC"]):
    p = predictions[(predictions.Group == GROUP) & (predictions.variant == VARIANT) & (predictions.target == target)]
    ax.scatter(p.observed, p.predicted, c=p.fold, cmap="viridis")
    lo, hi = min(p.observed.min(),p.predicted.min()), max(p.observed.max(),p.predicted.max())
    ax.plot([lo,hi],[lo,hi], "k--")
    ax.set(title=target, xlabel="Observed", ylabel="Future-month prediction")
fig.suptitle(f"{GROUP}: {VARIANT}; colors = temporal folds")
plt.show()

## Linear Regression và XGBoost theo nhóm 

Phần này dùng **5-fold shuffled end-to-end CV**: component và mask được fit riêng trên train của từng fold; held-out chỉ suy ra biên độ với component cố định. Khác với Ridge temporal ở trên, các fold ở đây được shuffle.

Feature cho một component là cực đại intensity đóng góp trên miền được vẽ:

\[
Fmax_{nk}=a_{nk}\max_{j\in\mathcal M}\phi_{kj}.
\]

- Fit **LinearRegression có intercept** và **XGBRegressor** cho từng component, từng target BOD/COD/TOC và từng nhóm, ở cả ba biến thể V0 cải tiến.
- Mỗi lần fit chỉ dùng **một Fmax của một peak component** với một target. Dùng nguyên thang, không log; loại background.
- KFold=5, shuffle=True, random_state=42; XGBoost: 80 cây, depth=2, learning_rate=0.05, min_child_weight=2, subsample=0.9, reg_lambda=2, hist và một thread, như notebook cũ.
- `R2_train`: fit và đánh giá trên toàn bộ dữ liệu nhóm. `R2_cv_mean ± R2_cv_std`: mean và sample SD giữa 5 fold. `R2_oof`: R² từ toàn bộ dự đoán held-out ghép lại, **không đồng nhất** với mean fold R².
- Chọn component tốt nhất theo CV mean cho từng **variant × nhóm × target × estimator**. Việc chọn và báo trên cùng CV có selection optimism; các bảng này dùng để khám phá, không thay thế temporal evaluation ở trên.
- Tên B/T/M/A/C/UV là tên vùng giả thuyết, không xác nhận danh tính hóa học.

Các file CSV/PNG/report mới nằm trong `component_regression/`, không ghi đè kết quả Ridge temporal.

In [ ]:
from evaluate_v0_component_regression import run_component_regression, XGB_PARAMS
REGRESSION_CV_SPLITS = 5
REGRESSION_SEED = 42
REGRESSION_VARIANT = "diag_soft_prior"  # hoặc diag_no_prior / rho_soft_prior
regression_results = run_component_regression(
    OUT, n_splits=REGRESSION_CV_SPLITS, seed=REGRESSION_SEED,
    xgb_params=XGB_PARAMS,
)
REGRESSION_OUT = regression_results["output"]
print("Protocol: train-only decomposition; end-to-end 5-fold shuffled CV")
print("Output:", REGRESSION_OUT)
display(regression_results["summary"].round(3))

### Component tốt nhất: R² train và R² CV của Linear/XGBoost

Hai estimator có thể chọn component khác nhau. Đổi `REGRESSION_VARIANT` để xem một biến thể khác; tất cả biến thể đã có trong CSV. Giá trị âm được giữ nguyên. Heatmap giới hạn màu dưới −1 nhưng nhãn luôn hiển thị giá trị R² thật.

In [ ]:
best_comparison = regression_results["comparison"]
selected_comparison = best_comparison[best_comparison.variant.eq(REGRESSION_VARIANT)].drop(columns="variant")
display(selected_comparison.round(4))
display(Image(filename=str(REGRESSION_OUT / f"{REGRESSION_VARIANT}_train_cv_r2.png")))
selected_best = regression_results["best"].query("variant == @REGRESSION_VARIANT")
best_readable = selected_best[["Group", "target", "model", "component", "R2_train", "R2_cv_mean", "R2_cv_std", "R2_oof"]].copy()
best_readable["CV R² ± SD"] = best_readable.apply(lambda r: f"{r.R2_cv_mean:.3f} ± {r.R2_cv_std:.3f}", axis=1)
display(best_readable.drop(columns=["R2_cv_mean", "R2_cv_std"]).sort_values(["Group", "target", "model"]).round(3))

### Đánh giá riêng từng peak component

Mỗi dòng là một mô hình **một feature peak → một target**, fit riêng cho một nhóm. Bảng thứ nhất xem sáu peak riêng rẽ tại `GROUP`; bảng thứ hai xem peak được chọn (`PEAK_COMPONENT`, mặc định T) trên cả các nhóm đủ dữ liệu. Đổi biến này thành B/M/A/C/UV để xem peak khác. Toàn bộ kết quả của cả sáu peak luôn có trong CSV.

Cột `n_features=1` xác nhận mỗi mô hình chỉ dùng một feature. `status` ghi trường hợp không đủ dữ liệu hoặc feature/target hằng.

In [ ]:
all_scores = regression_results["scores"]
PEAK_COMPONENT = "T"  # B, T, M, A, C hoặc UV
columns = ["Group", "target", "model", "component", "n_features", "n", "R2_train", "R2_cv_mean", "R2_cv_std", "R2_oof", "status"]
detailed = all_scores[all_scores.component.ne("all_components") & (all_scores.variant == REGRESSION_VARIANT) & (all_scores.Group == GROUP)]
print(f"Từng peak riêng rẽ tại {GROUP} — mỗi model có một feature")
display(detailed[columns].sort_values(["target", "model", "component"]).round(4))
selected_peak = all_scores[(all_scores.variant == REGRESSION_VARIANT) & (all_scores.component == PEAK_COMPONENT)]
print(f"Chỉ peak {PEAK_COMPONENT} → target, đánh giá riêng trên các nhóm đủ dữ liệu")
display(selected_peak[columns].sort_values(["Group", "target", "model"]).round(4))
print("Statuses:", all_scores.status.value_counts().to_dict())
print("Constant target test folds:", int(all_scores.constant_test_folds.sum()))
assert set(all_scores.Group) == set(group_manifest.Group)
assert set(all_scores.model) == {"linear", "xgb"}
assert set(all_scores.component) == {"B", "T", "M", "A", "C", "UV", "all_components"}
assert all_scores.loc[all_scores.component.ne("all_components"), "n_features"].eq(1).all()
print("Report:", REGRESSION_OUT / "report.md")

## Kiểm tra protocol và kết luận tự động

Không báo “peak sinh hóa đúng” chỉ vì tâm nằm trong prior. Cần dữ liệu chuẩn/replicate, metadata scatter và nhiều split độc lập hơn để xác nhận. Các run chưa hội tụ hoặc peak không ổn định được giữ trong bảng để đánh giá trung thực.

In [ ]:
folds = pd.read_csv(OUT / "folds.csv")
for (group, fold), block in folds.groupby(["Group", "fold"]):
    train = block[block.role == "train"]
    test = block[block.role == "test"]
    assert set(train.sample_index).isdisjoint(set(test.sample_index))
    assert (train.year*12+train.month).max() < (test.year*12+test.month).min()
for (group, variant, target), p in predictions.groupby(["Group", "variant", "target"]):
    assert not p.sample_index.duplicated().any()
assert len(folds.Group.unique()) == len(group_manifest)
coverage = pd.read_csv(OUT / "reconstruction.csv")
print(f"Temporal split checks passed for {len(group_manifest)} groups.")
print("Max held-out energy fraction omitted by training mask:",
      coverage[coverage.split.ne("full")].unseen_energy_fraction.max())
print("Median in-sample improvement vs old V0 (%):", round(comparison.relative_improvement_pct.median(), 2))
print("Groups helped by prior on future EEM reconstruction:", int((ablation.prior_minus_no_prior<0).sum()), f"/ {len(group_manifest)}")
print("Groups helped by limited rho on future EEM reconstruction:", int((ablation.rho_minus_diag<0).sum()), f"/ {len(group_manifest)}")
print("Artifacts:", OUT)
print((OUT / "report.md").read_text())

## Hướng thử tiếp theo

- Tăng ngân sách cho các nghiệm chưa hội tụ; chạy thêm seed và phân chia độc lập để xác nhận độ ổn định.
- Tách tác động rank bằng thí nghiệm 4/5/6/7 component cùng protocol; không chọn bằng target test.
- Thử phổ Ex nhiều lobes dùng chung Em/amplitude, sau khi Gaussian diagonal ổn định.
- Xác minh mask vật lý, blank, hiệu chỉnh inner-filter và Raman normalization trước diễn giải định lượng sinh hóa.
- Kiểm tra trên dữ liệu tổng hợp bất đối xứng/phi Gaussian và chuẩn thực nghiệm; sanity check ở trên chỉ là bước đầu.

Nguồn: [báo cáo audit nội bộ](../docs/v0_peak_model_review.md), [Murphy et al. (2013)](https://doi.org/10.1039/C3AY41160E), [định nghĩa peak eemR](https://pmassicotte.github.io/eemR/reference/eem_coble_peaks.html).